# Offline Reproducible Evaluation: v2 Baseline vs. v3.2 Upgrade

**Candidate:** Ruhaan Kakar  
**Prepared for:** Dr. Jagadeeswara Reddy Devasani, Loopworm Private Limited  
**Project:** Mini Literature Classification Engine (`v3.2` — Criterion-Aware Multi-Condition, 3-Class, Real MLE Calibration)

- **Environment:** Python 3.11 | PyTorch 2.6.0+cu124 | `cross-encoder/nli-deberta-v3-small` (rev `fa280487`)
- **Tags:** `v2-baseline` (`85894a0`) vs. `v3.2-frozen`
- Runs 100% offline against SHA-256 audited snapshots (`data/snapshots/manifest.json`) using real `DebertaNLIScorer`.

In [1]:
import hashlib
import json
import os
import sys
import pandas as pd

BASE_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

from app.nli import DebertaNLIScorer
from app.scoring import load_calibration_config, reload_calibration_config
from app.text_utils import init_punkt
from eval.metrics import mcnemar_exact
from eval.run import (
    load_splits,
    load_dataset,
    audit_snapshot_manifest,
    compute_double_label_kappa,
    evaluate_rows,
)

init_punkt()
splits = load_splits()
rows = load_dataset(stratum_filter="all")
print(f"Loaded {len(rows)} benchmark rows.")
print("Dev entities:", splits["dev_entities"])
print("Test entities:", splits["test_entities"])

Loaded 152 benchmark rows.
Dev entities: ['vanillin', 'caffeine', 'curcumin', 'chitin']
Test entities: ['palladium(II) acetate', 'artemisinic acid', 'lipase']


In [2]:
# Cell 2: Snapshot SHA-256 drift audit (WS13-B)
audit = audit_snapshot_manifest(rows)
print("Snapshot Manifest Audit PASSED:", audit)
kappa_noisy = compute_double_label_kappa([r for r in rows if r.get("stratum") == "noisy"])
print("Noisy Real-World Stratum Inter-Annotator Agreement:", kappa_noisy)

Snapshot Manifest Audit PASSED: {'audited_records': 152, 'manifest_version': 'v3.2'}
Noisy Real-World Stratum Inter-Annotator Agreement: {'n_double_labeled': 11, 'exact_agreement': 0.9091, 'cohens_kappa': 1.0}


In [3]:
# Cell 3 & 4: Execute v2 baseline vs. v3.2 using real DebertaNLIScorer (no stubs)
reload_calibration_config()
calib_cfg = load_calibration_config(strict_splits=True)
scorer = DebertaNLIScorer()

test_rows = [r for r in rows if r["entity"] in set(splits["test_entities"])]
print(f"Evaluating on {len(test_rows)} held-out test papers ({splits['test_entities']})...")

res_v2 = evaluate_rows(test_rows, scorer=scorer, pipeline="v2", strict_verification=False)
res_v3 = evaluate_rows(test_rows, scorer=scorer, pipeline="v3", strict_verification=True)

with open(os.path.join(BASE_DIR, "eval", "results_v2.json"), "w", encoding="utf-8") as f:
    json.dump(res_v2, f, indent=2)
with open(os.path.join(BASE_DIR, "eval", "results_v3.json"), "w", encoding="utf-8") as f:
    json.dump(res_v3, f, indent=2)
print("Completed v2 and v3.2 test evaluations cleanly.")

Evaluating on 56 held-out test papers (['palladium(II) acetate', 'artemisinic acid', 'lipase'])...


Completed v2 and v3.2 test evaluations cleanly.


In [4]:
# Cell 5 & 6: 3-Class Confusion Matrices, Stratification (Clean vs Noisy), Wilson CIs
print("--- v2 3x3 Confusion Matrix (mapped from binary) ---")
print(pd.DataFrame(res_v2["multiclass"]["confusion_matrix"]).T)
print("\n--- v3.2 3x3 Confusion Matrix ---")
print(pd.DataFrame(res_v3["multiclass"]["confusion_matrix"]).T)

def summarize(res, name):
    mc = res["multiclass"]
    q = mc["per_class"]["qualifies"]
    return {
        "Model": name,
        "N": mc["n"],
        "Qualifies Prec": round(q["precision"], 4),
        "Qualifies Rec": round(q["recall"], 4),
        "FP / FN (Qual)": f"{mc['qualifies_fp']} / {mc['qualifies_fn']}",
        "Macro F1": round(mc["macro_f1"], 4),
        "Abstention Rate": round(mc["abstention_rate"], 4),
        "Abstention Err": round(mc["abstention_error_rate"], 4),
        "ECE": res["ece_calibrated"] if "v3" in name else res["ece_raw_baseline"],
    }

summary_df = pd.DataFrame([summarize(res_v2, "v2-baseline"), summarize(res_v3, "v3.2-frozen")])
print("\n--- Held-Out Test Comparison ---")
print(summary_df.to_string(index=False))

print("\n--- Stratified Clean vs Noisy Real Europe PMC (v3.2) ---")
for st, sm in res_v3["by_stratum"].items():
    print(f"  {st}: n={sm['n']}, macro_f1={sm['macro_f1']:.4f}, acc={sm['accuracy']:.4f}, abstain_rate={sm['abstention_rate']:.4f}, qualifies_fp={sm['qualifies_fp']}")

--- v2 3x3 Confusion Matrix (mapped from binary) ---
                       qualifies  does_not_qualify  insufficient_evidence
qualifies                     19                 0                      0
does_not_qualify              10                24                      0
insufficient_evidence          2                 1                      0

--- v3.2 3x3 Confusion Matrix ---
                       qualifies  does_not_qualify  insufficient_evidence
qualifies                     17                 0                      2
does_not_qualify               0                26                      8
insufficient_evidence          1                 1                      1

--- Held-Out Test Comparison ---
      Model  N  Qualifies Prec  Qualifies Rec FP / FN (Qual)  Macro F1  Abstention Rate  Abstention Err    ECE
v2-baseline 56          0.6129         1.0000         12 / 0    0.5245           0.0000          0.3333 0.3340
v3.2-frozen 56          0.9444         0.8947          1 / 2    

In [5]:
# Cell 7: Exact McNemar Significance Test (3-class & binary qualifies slice)
yt, p2, p3 = res_v3["y_true_3"], res_v2["y_pred_3"], res_v3["y_pred_3"]
b3 = sum(1 for t, a, c in zip(yt, p3, p2) if a == t and c != t)
c3 = sum(1 for t, a, c in zip(yt, p3, p2) if a != t and c == t)
p_mc_3 = mcnemar_exact(b3, c3)

yb = [1 if t == "qualifies" else 0 for t in yt]
b_bin = sum(1 for t, a, c in zip(yb, [1 if x == 'qualifies' else 0 for x in p3], [1 if x == 'qualifies' else 0 for x in p2]) if a == t and c != t)
c_bin = sum(1 for t, a, c in zip(yb, [1 if x == 'qualifies' else 0 for x in p3], [1 if x == 'qualifies' else 0 for x in p2]) if a != t and c == t)
p_mc_b = mcnemar_exact(b_bin, c_bin)

print(f"McNemar 3-class: b={b3}, c={c3}, p={p_mc_3:.4f}")
print(f"McNemar binary qualifies: b={b_bin}, c={c_bin}, p={p_mc_b:.4f}")

McNemar 3-class: b=6, c=5, p=1.0000
McNemar binary qualifies: b=11, c=2, p=0.0225


In [6]:
# Cell 8: Calibration Audit (Platt MLE config, ECE before vs after, D5 null-confidence audit)
print("Calibration Config (config/calibration.json):")
print(json.dumps(calib_cfg, indent=2))
print(f"\nECE Raw Baseline: {res_v2['ece_raw_baseline']:.4f} -> ECE v3.2 Calibrated: {res_v3['ece_calibrated']:.4f}")
print("Reliability Bins:", pd.DataFrame(res_v3["ece_bins"]).to_string(index=False))

Calibration Config (config/calibration.json):
{
  "calibration_version": "platt-v3.2-mle",
  "fitted_at": "2026-11-12T16:01:42.260903+00:00",
  "method": "platt_scaling_logistic_mle",
  "splits_sha256": "2e7d2ea53f19830d3971ab11922770c5d0ecc8f13cf8ae7bc3a4e1924858bf09",
  "data_hash": "852cdbd07bb1fd9be5ce2b8a3e60f522016c3354f5e07f9bc367b54a1127da87",
  "fit_entities": [
    "vanillin",
    "caffeine",
    "curcumin",
    "chitin"
  ],
  "neg_fit_window": 0.15,
  "qualifier_path": {
    "fitted": true,
    "A": 3.459395,
    "B": -1.162682,
    "n_total": 33,
    "n_pos": 28,
    "n_neg": 5,
    "out_of_fold_brier": 0.1195,
    "loo_mean_A": 3.4353,
    "loo_std_A": 0.8332,
    "slope_unstable_flag": false
  },
  "negative_path": {
    "fitted": true,
    "A": 1.549835,
    "B": -0.355466,
    "n_rows": 4,
    "out_of_fold_brier": 0.1897,
    "reason_codes_included": [
      "different_activity",
      "wrong_system"
    ]
  }
}

ECE Raw Baseline: 0.3340 -> ECE v3.2 Calibrated: 0.0695


In [7]:
# Cell 9 & 10: Case Studies (v2 false positives fixed by v3.2) & Threshold Sensitivity Check
fixed_cases = [
    (t2, t3)
    for t2, t3 in zip(res_v2["traces"], res_v3["traces"])
    if t2["pred"] != t2["gold"] and t3["pred"] == t3["gold"]
]
print(f"Fixed Errors (v2 wrong -> v3.2 right): {len(fixed_cases)} papers")
for t2, t3 in fixed_cases[:5]:
    print(f"  [{t3['entity']}] {t3['paper_id']} ({t3['stratum']}): gold={t3['gold']} | v2={t2['pred']} -> v3.2={t3['pred']} ({t3['reason_code']})")

print("\n--- LOEO Threshold Sensitivity Check (+-20% around tau=0.45) on Test Fold ---")
for t_val in [0.36, 0.45, 0.54]:
    s_res = evaluate_rows(test_rows, scorer=scorer, pipeline="v3", tau=t_val, strict_verification=True)
    mc = s_res["multiclass"]
    print(f"  tau={t_val:.2f}: macro_f1={mc['macro_f1']:.4f}, qualifies_fp={mc['qualifies_fp']}, abstain_rate={mc['abstention_rate']:.4f}")

Fixed Errors (v2 wrong -> v3.2 right): 6 papers
  [lipase] 10.21203/rs.3.rs-9942287/v1 (noisy): gold=does_not_qualify | v2=qualifies -> v3.2=does_not_qualify (reagent_role)
  [lipase] 42778850 (noisy): gold=does_not_qualify | v2=qualifies -> v3.2=does_not_qualify (review_article)
  [artemisinic acid] 10.21203/rs.3.rs-3926665/v1 (noisy): gold=does_not_qualify | v2=qualifies -> v3.2=does_not_qualify (wrong_system)
  [artemisinic acid] 42418950 (noisy): gold=insufficient_evidence | v2=does_not_qualify -> v3.2=insufficient_evidence (system_unstated)
  [artemisinic acid] 42774055 (noisy): gold=does_not_qualify | v2=qualifies -> v3.2=does_not_qualify (wrong_system)

--- LOEO Threshold Sensitivity Check (+-20% around tau=0.45) on Test Fold ---


  tau=0.36: macro_f1=0.6474, qualifies_fp=2, abstain_rate=0.1607


  tau=0.45: macro_f1=0.6381, qualifies_fp=1, abstain_rate=0.1964


  tau=0.54: macro_f1=0.6249, qualifies_fp=1, abstain_rate=0.2143


In [8]:
# Cell 11: Final Deterministic Artifact Digests
for fname in ["eval/results_v2.json", "eval/results_v3.json", "config/calibration.json"]:
    fpath = os.path.join(BASE_DIR, fname)
    h = hashlib.sha256(open(fpath, "rb").read()).hexdigest()
    print(f"{fname}: {h[:16]}...")

eval/results_v2.json: 40679c2af266a225...
eval/results_v3.json: b66f56d99df5830a...
config/calibration.json: 0116b429e9d4c3c7...
